# LSST SN Ia Single-Object Simulation

Forward-model **one** SN Ia light curve at a user-specified sky position and SALT3 parameters
against a DP2 CCDVisit/detector table with `lightcurvelynx`, instead of sampling a population.

**Survey**: DP2 visit-detector table (`DP2_VISIT_DETECTOR_FILE`, the real DP2 `visit_detector.parquet`)  
**Model**: SALT3 via `SncosmoWrapperModel`  
**Filters**: u, g, r, i, z, y  
**Parameters**: fixed RA, Dec, t0, x0, x1, c, and redshift (`OBJECT_PARAMS`) — not sampled.
Set `NUM_REALIZATIONS > 1` to draw multiple independent noise realizations of the same object.

## 1. Imports

## 0. Environment Setup

Set `LIGHTCURVELYNX_DATA_DIR` **before** importing lightcurvelynx — the download path is resolved at import time.  
Downloaded files (OpSim DB, passbands) will be stored in `./data/` inside this project directory.

In [ ]:
# auto reload
%load_ext autoreload
%autoreload 2

In [ ]:
# %pip install sfdmap2
# %pip install iminuit

In [ ]:
# # setup dustmaps, run once
# from dustmaps.config import config
# config.reset()

In [ ]:
import os
from pathlib import Path

# Store downloaded data inside this project so it travels with the repo checkout.
# Must be set before any lightcurvelynx imports (path is resolved at import time).
_data_dir = Path().resolve() / "data"
_data_dir.mkdir(exist_ok=True)
os.environ["LIGHTCURVELYNX_DATA_DIR"] = str(_data_dir)
print(f"LIGHTCURVELYNX_DATA_DIR = {_data_dir}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2
from nested_pandas import read_parquet
from joblib.externals.loky import get_reusable_executor
import sncosmo
from astropy.cosmology import Planck18
from astropy.coordinates import Distance

from lightcurvelynx.obstable.lsst_obstable import LSSTObsTable
from lightcurvelynx.astro_utils.passbands import PassbandGroup
from lightcurvelynx.models.sncosmo_models import SncosmoWrapperModel
from lightcurvelynx.simulate import simulate_lightcurves
from lightcurvelynx.utils.extrapolate import LinearDecayOnMag,ZeroPadding
from lightcurvelynx.astro_utils.dustmap import DustmapWrapper,SFDMap
from lightcurvelynx.effects.extinction import ExtinctionEffect
from lightcurvelynx.astro_utils.mag_flux import flux2mag

import lightcurvelynx
print(lightcurvelynx.__version__)

## 2. Simulation Configuration

In [ ]:
SEED = 1024
RNG  = np.random.default_rng(SEED)

SKIP_SIM = True   # Set to True to skip simulation and load existing results from disk.
SKIP_LCFIT = False # Set to True to skip LC fitting and load existing results from disk.
FIT_Z = True # Fit redshift in SALT fit.
FIT_Z_SUFFIX = "_fitz" if FIT_Z else ""

# The DP2 CCDVisit/detector table has no field/program column, so DDF/WFD visits can't be
# selected by string match (unlike the opsim `observation_reason` column used in
# dp2_sims_ddf.ipynb). Instead, select/exclude by proximity to known DDF field centers.
SELECT_DDF_ONLY = False  # Set to True to keep only visits within DDF_RADIUS_DEG of a DDF field.
SELECT_WFD_ONLY = False  # Set to True to exclude visits within DDF_RADIUS_DEG of a DDF field.
assert not (SELECT_DDF_ONLY and SELECT_WFD_ONLY), "SELECT_DDF_ONLY and SELECT_WFD_ONLY are mutually exclusive."

OUTPUT_SUFFIX = "_ddf" if SELECT_DDF_ONLY else "_wfd" if SELECT_WFD_ONLY else ""  # Tag output filenames.

SIM_PARAMS = {
    "filters": ["u", "g", "r", "i", "z", "y"],
}

NUM_REALIZATIONS = 1  # increase to draw multiple independent noise realizations of the same object

# Edit the values below to specify the SN Ia to simulate. RA/Dec must fall within the loaded
# visit table's footprint (see Section 3's coverage/footprint plot), and t0 should fall within
# the survey's MJD range (see Section 3's printed range), for any observations to be found.
OBJECT_PARAMS = {
    "ra": 150.1167,      # deg — placeholder (COSMOS field, known within DP2 footprint); edit as needed
    "dec": 2.2058,       # deg
    "t0": 60900.0,       # MJD — placeholder; edit to fall within the survey's MJD range
    "x0": 1.0e-5,        # SALT3 amplitude — placeholder
    "x1": 0.0,
    "c": 0.0,
    "redshift": 0.1,
}

## 3. Load LSST Visit Table

Loading the DP2 visit-detector table from `DP2_VISIT_DETECTOR_FILE`.

If `SELECT_DDF_ONLY` is `True`, visits are restricted to those within `DDF_RADIUS_DEG` of a
known Deep Drilling Field center (COSMOS, ECDFS, EDFS a/b, ELAIS-S1, XMM-LSS). If
`SELECT_WFD_ONLY` is `True`, those DDF visits are excluded instead, leaving only the wide-fast-deep
survey. The CCDVisit table has no `observation_reason`/field column to filter on directly, so
this selection is done by sky position instead.

In [ ]:
DP2_VISIT_DETECTOR_FILE = "/rubin/lsdb_data/dp2/public-files/visit_detector.parquet"
dp2_visit_table = pd.read_parquet(DP2_VISIT_DETECTOR_FILE)

# LSST Deep Drilling Field centers (rubin_scheduler / rubin_sim definitions).
DDF_FIELDS = {
    "COSMOS":   (150.1167, 2.2058),
    "ECDFS":    (53.125,  -28.100),
    "EDFS_a":   (58.90,   -49.315),
    "EDFS_b":   (63.60,   -47.60),
    "ELAIS-S1": (9.45,    -44.00),
    "XMM-LSS":  (35.708,  -4.750),
}
DDF_RADIUS_DEG = 1.75  # matching radius around each DDF field center

def select_ddf_visits(visit_table, ra_col="ra", dec_col="dec"):
    from astropy.coordinates import SkyCoord
    import astropy.units as u

    visit_coords = SkyCoord(ra=visit_table[ra_col].values * u.deg, dec=visit_table[dec_col].values * u.deg)
    is_ddf = np.zeros(len(visit_table), dtype=bool)
    for ra, dec in DDF_FIELDS.values():
        field_coord = SkyCoord(ra=ra * u.deg, dec=dec * u.deg)
        is_ddf |= visit_coords.separation(field_coord).deg < DDF_RADIUS_DEG
    return is_ddf

if SELECT_DDF_ONLY:
    n_before = len(dp2_visit_table)
    dp2_visit_table = dp2_visit_table[select_ddf_visits(dp2_visit_table)]
    print(f"Selected {len(dp2_visit_table):,} DDF visits out of {n_before:,} total")
elif SELECT_WFD_ONLY:
    n_before = len(dp2_visit_table)
    dp2_visit_table = dp2_visit_table[~select_ddf_visits(dp2_visit_table)]
    print(f"Selected {len(dp2_visit_table):,} WFD visits out of {n_before:,} total (excluded DDF)")

obstable = LSSTObsTable.from_ccdvisit_table(dp2_visit_table, make_detector_footprint=True)
print(f"Visit table loaded: {len(obstable):,} observations")
print(f"MJD range: {obstable['time'].min():.1f} – {obstable['time'].max():.1f}")
print(f"Filters:    {sorted(obstable['filter'].unique())}")

In [ ]:
obstable.head()

In [ ]:
sky_coverage = obstable.estimate_coverage(max_depth=8,use_footprint=False)
print(f"Estimated sky coverage: {sky_coverage:.0f} deg²  (informational only — check OBJECT_PARAMS ra/dec falls within this)")

In [ ]:
obstable.plot_footprint(depth=8,use_footprint=False)

In [ ]:
t_min = float(obstable["time"].min())
t_max = float(obstable["time"].max())
print(f"Survey MJD range: {t_min:.1f} – {t_max:.1f}")
print(f"Object RA/Dec/t0: {OBJECT_PARAMS['ra']:.4f}, {OBJECT_PARAMS['dec']:.4f}, {OBJECT_PARAMS['t0']:.1f}")
if not (t_min <= OBJECT_PARAMS["t0"] <= t_max):
    print("WARNING: OBJECT_PARAMS['t0'] falls outside the survey MJD range — the light curve may be empty.")

## 4. Load LSST Passbands

In [ ]:
passbands = PassbandGroup.from_preset("LSST", filters=SIM_PARAMS["filters"])
print(passbands)

## 5. Build SN Ia Source Model

All SALT3 parameters and the sky position are **fixed floats** taken directly from `OBJECT_PARAMS`,
not sampled:
- **RA, Dec, t0, x0, x1, c, redshift** — fixed values from `OBJECT_PARAMS`

Dust extinction (Milky Way, SFD map) is still computed dynamically at the fixed `(ra, dec)` via
`DustmapWrapper`/`ExtinctionEffect`.

In [ ]:
# Extrapolation settings
time_extrap_before = ZeroPadding()
time_extrap_after = LinearDecayOnMag(decay_rate=0.02, mag_thres=30.)
wave_extrap_before = ZeroPadding()
wave_extrap_after = ZeroPadding()

# Assemble the SALT3 source (no host galaxy) with fixed parameters
source = SncosmoWrapperModel(
    "salt3",
    t0=OBJECT_PARAMS["t0"],
    x0=OBJECT_PARAMS["x0"],
    x1=OBJECT_PARAMS["x1"],
    c=OBJECT_PARAMS["c"],
    ra=OBJECT_PARAMS["ra"],
    dec=OBJECT_PARAMS["dec"],
    redshift=OBJECT_PARAMS["redshift"],
    node_label="source",
    time_extrapolation=(time_extrap_before, time_extrap_after),
    wave_extrapolation=(wave_extrap_before, wave_extrap_after),
)

from dustmaps.sfd import SFDQuery
mwextinction = DustmapWrapper(SFDQuery(), ra=source.ra, dec=source.dec, node_label="mwext")
ext_effect = ExtinctionEffect(extinction_model="F99", ebv=mwextinction,
                              r_v=3.1, frame='observer', backend="dust_extinction")
source.add_effect(ext_effect)

print("Source model built successfully.")

## 6. Run Simulation

In [ ]:
if not SKIP_SIM:
    param_cols = [
        "source.t0",
        "source.x0",
        "source.x1",
        "source.c",
        "source.redshift",
        "source.ra",
        "source.dec",
    ]
    obstable_save_cols = ["zp"]

    NJOBS = 8
    BATCH_SIZE = 3000
    executor = get_reusable_executor(max_workers=4)
    lightcurves = simulate_lightcurves(
        model=source,
        num_samples=NUM_REALIZATIONS,
        survey_info=obstable,
        passbands=passbands,
        param_cols=param_cols,
        obstable_save_cols=obstable_save_cols,
        rng=RNG,
        num_jobs=NJOBS,
        batch_size=BATCH_SIZE,
        executor=executor,
    )
    print(f"Simulated {len(lightcurves):,} realization(s)")
    lightcurves.head()

## 7. Save Results

In [ ]:
if not SKIP_SIM:
    from pathlib import Path
    output_path = Path(f"outputs/lsst_snia_dp2_singleobject{OUTPUT_SUFFIX}_results.parquet")
    output_path.parent.mkdir(exist_ok=True)
    lightcurves.to_parquet(output_path)
    print(f"Saved to {output_path}")

In [ ]:
lightcurves = read_parquet(f"outputs/lsst_snia_dp2_singleobject{OUTPUT_SUFFIX}_results.parquet")
lightcurves = lightcurves.drop(columns=["params"]).dropna(subset=['lightcurve'])
print(f"Loaded {len(lightcurves):,} realization(s) with a valid lightcurve")

## 8. Diagnostics

Inspect the simulated light curve(s) and fit them back with SALT3 to check parameter recovery.

In [ ]:
results = lightcurves

In [ ]:
# example light curve for the (first) simulated realization
sn = results.iloc[0]
lc = sn["lightcurve"]
print(lc)
for band in SIM_PARAMS["filters"]:
    mask = lc["filter"] == band
    mask &= (lc["mjd"] - sn["source_t0"])/(1. + sn["source_redshift"]) > -20  # only show points within 20 days before t0
    mask &= (lc["mjd"] - sn["source_t0"])/(1. + sn["source_redshift"]) < 100   # only show points within 20 days after t0
    if mask.any():
        plt.errorbar(lc["mjd"][mask], lc["flux"][mask], lc["fluxerr"][mask],
                     fmt="o", label=band, capsize=3)
plt.axvline(sn["source_t0"], ls="--", color="k", label="t0")
plt.legend()
plt.xlabel("MJD")
plt.ylabel("Flux (nJy)")
plt.title(f'SN Ia  z={sn["source_redshift"]:.3f}')
plt.show()

In [ ]:
def infer_z_from_peakmag(row):
    peakflux = np.max(row)
    peakmag = flux2mag(peakflux)
    z = Distance(distmod = peakmag + 19).compute_z(cosmology=Planck18).value
    return z

In [ ]:
# append system path 

from utils.lcfit import fit_single_lc  

def fit_single_lc_w_cond(lc,
                         bounds={"x1": (-4,4),
                                 "c": (-0.4,0.4),},
                         phase_range=(-10,40),
                         modelcov=False):
    return fit_single_lc(lc,mpbounds=bounds,phase_range=phase_range,modelcov=modelcov)

In [ ]:
lc_to_fit = results.iloc[0:]

In [ ]:
lc_to_fit = lc_to_fit.map_rows(infer_z_from_peakmag, columns=["lightcurve.flux"], row_container="args",
                     output_names=["z_est"], append_columns=True)

In [ ]:
res = fit_single_lc_w_cond(lc_to_fit.iloc[0])
res

In [ ]:
%%time
if not SKIP_LCFIT:
    executor = get_reusable_executor(max_workers=min(NUM_REALIZATIONS, 4), kill_workers=True)
    futures = [executor.submit(fit_single_lc_w_cond, row) for _index, row in lc_to_fit.iterrows()]
    fit_results = [f.result() for f in futures]
    result_df = pd.DataFrame(fit_results)

In [ ]:
if not SKIP_LCFIT:
    result_df.to_csv(f"outputs/lsst_snia_dp2_singleobject{OUTPUT_SUFFIX}_lcfit_results{FIT_Z_SUFFIX}.csv", index=True)
    print(f"Saved {len(result_df)} LC fit results")

In [ ]:
result_df = pd.read_csv(f"outputs/lsst_snia_dp2_singleobject{OUTPUT_SUFFIX}_lcfit_results{FIT_Z_SUFFIX}.csv")

In [ ]:
# Compare the SALT3 fit against the true (fixed) input parameters
print("Fit vs. true parameters:")
for i, row in result_df.iterrows():
    print(f"  Realization {i}: "
          f"x1={row.x1:.3f} (true {OBJECT_PARAMS['x1']:.3f}), "
          f"c={row.c:.3f} (true {OBJECT_PARAMS['c']:.3f}), "
          f"z={row.z:.3f} (true {OBJECT_PARAMS['redshift']:.3f})")
result_df

In [ ]:
# plot all simulated light curves (up to max_number)
max_number = 10
for i in range(min(len(result_df), max_number)):
    sn = results.loc[results.id == int(result_df.iloc[i].id)]
    print(sn)
    lc = sn["lightcurve"]
    for band in SIM_PARAMS["filters"]:
        mask = lc["filter"] == band
        mask &= (lc["mjd"] - sn["source_t0"])/(1. + sn["source_redshift"]) > -20  # only show points within 20 days before t0
        mask &= (lc["mjd"] - sn["source_t0"])/(1. + sn["source_redshift"]) < 100   # only show points within 20 days after t0
        if mask.any():
            plt.errorbar(lc["mjd"][mask], lc["flux"][mask], lc["fluxerr"][mask],
                        fmt="o", label=band, capsize=3)
    plt.axvline(sn["source_t0"].values[0], ls="--", color="k", label="t0")
    plt.legend()
    plt.xlabel("MJD")
    plt.ylabel("Flux (nJy)")
    plt.title(f'SN Ia  z={sn["source_redshift"].values[0]:.3f}')
    plt.show()